# 阿里 PPU：DolphinDB 分钟数据全量内存训练

本 Notebook 从 DolphinDB 按交易日并发读取分钟数据，直接写入 PPU 节点 RAM。训练开始后不再访问 DolphinDB，也不生成分钟 MemMap 文件。700GB 节点默认预留 100GB，基础分钟数组上限设为 550GB。

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    raise FileNotFoundError('请先进入 AlphaMining-GFlowNet-AlphaEval 仓库根目录')
print('PROJECT_ROOT =', PROJECT_ROOT)

## 安装依赖

安装完成后如 Jupyter 提示重启 Kernel，请重启后从下一单元格继续。

In [ ]:
%pip install -q -r requirements.txt
%pip install -q -r requirements-ddb.txt

## 检查 DDB 环境变量

请在启动 JupyterLab 前设置这些环境变量。Notebook 只检查是否存在，不会打印账号或密码。

In [ ]:
required_env = [
    'DDB_HOST', 'DDB_PORT', 'DDB_USER', 'DDB_PASSWORD',
    'DDB_DATABASE', 'DDB_TABLE', 'DDB_TRADE_DAYS_DATABASE',
]
missing = [name for name in required_env if not os.environ.get(name)]
if missing:
    raise EnvironmentError('缺少环境变量: ' + ', '.join(missing))
print('DDB环境变量检查通过；敏感值未显示')

## 检查内存与配置

正式加载前确认可用内存接近 700GB，并把配置中的 `prices_are_adjusted` 改为真实状态。

In [ ]:
import os
import yaml

CONFIG_PATH = Path('configs/minute_training_ppu_ddb_ram.yaml')
config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
total_ram = os.sysconf('SC_PHYS_PAGES') * os.sysconf('SC_PAGE_SIZE')
available_ram = os.sysconf('SC_AVPHYS_PAGES') * os.sysconf('SC_PAGE_SIZE')
print('total RAM GB     =', round(total_ram / 1024**3, 1))
print('available RAM GB =', round(available_ram / 1024**3, 1))
print('load_mode        =', config['dataset']['dolphindb']['load_mode'])
print('build_workers    =', config['dataset']['memory']['build_workers'])
print('reward_workers   =', config['dataset']['memory']['workers'])
assert config['dataset']['memory']['reward_parallel_backend'] == 'threading'

## 开始加载并训练

第一阶段日志为 `[DDBRAM]`，会显示容量估算、年度加载和驻留内存。出现 `load_complete` 后，DDB 连接会关闭并开始 GFlowNet 训练。停止 Kernel 会释放全部分钟 RAM 数据，下次启动需要重新从 DDB 加载。

In [ ]:
%run scripts/train_cpu.py --mode minute --config configs/minute_training_ppu_ddb_ram.yaml

## 输出

模型保存到 `checkpoints/gflownet_minute_ppu_ddb_ram_best.pt`，训练日志、指标、Alpha Pool 和因子矩阵保存在 `results/minute_ppu_ddb_ram/`。